## Kaggle Setup And Experimental Scope
This notebook consolidates the complete TE-Q-Transformer pipeline into a single reviewer-ready workflow for Kaggle execution on a T4 GPU. The setup cell below installs dependencies, defines the dataset and output paths, activates CUDA, and establishes the environment used for all subsequent data, training, evaluation, and export steps.

In [ ]:
!pip -q install pennylane scikit-learn seaborn matplotlib

import json
import os
import random
import shutil
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pennylane as qml
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from torch import nn, optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset


def seed_everything(seed: int = 42) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        if hasattr(torch.backends.cuda, 'enable_flash_sdp'):
            torch.backends.cuda.enable_flash_sdp(False)
        if hasattr(torch.backends.cuda, 'enable_mem_efficient_sdp'):
            torch.backends.cuda.enable_mem_efficient_sdp(False)
        if hasattr(torch.backends.cuda, 'enable_math_sdp'):
            torch.backends.cuda.enable_math_sdp(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=False)


seed_everything(42)

BATCH_TO_CELLS = {
    0: ('B0005', 'B0006', 'B0007', 'B0018'),
    1: ('B0029', 'B0030', 'B0031', 'B0032'),
    2: ('B0053', 'B0054', 'B0055', 'B0056'),
}
HYBRID_REQUIRED_CELLS = (
    'B0005', 'B0006', 'B0007',
    'B0018',
    'B0029', 'B0030', 'B0031', 'B0032',
    'B0053',
)
RAW_KAGGLE_HINT = '/kaggle/input/datasets/zadidallisan/quantum-sample/nasa'
REQUIRED_FILES = [
    f'{cell_id}_X.npy'
    for cell_id in HYBRID_REQUIRED_CELLS
] + [
    f'{cell_id}_soh.npy'
    for cell_id in HYBRID_REQUIRED_CELLS
]
OPTIONAL_FILES = [
    f'{cell_id}_X.npy'
    for cell_id in ('B0054', 'B0055', 'B0056')
] + [
    f'{cell_id}_soh.npy'
    for cell_id in ('B0054', 'B0055', 'B0056')
]
SAVE_DIR = Path('/kaggle/working/nasa_results/')
FIGURES_DIR = Path('/kaggle/working/nasa_figures/')
PREPROCESS_DIR = Path('/kaggle/working/nasa_preprocessed/')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESS_DIR.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU is not enabled. Please switch the notebook accelerator to T4 GPU.')
DEVICE = torch.device('cuda')

warnings.filterwarnings(
    'ignore',
    message='enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True',
)


def _looks_like_nasa_dir(path: Path) -> bool:
    return path.exists() and path.is_dir() and all((path / name).exists() for name in REQUIRED_FILES)


def _candidate_paths_from_hint(raw_hint: str) -> List[Path]:
    hint = Path(raw_hint)
    candidates = [hint]

    # Kaggle dataset web paths often include /datasets/<owner>/..., but mounted runtime
    # paths typically start at /kaggle/input/<dataset-slug>/...
    parts = list(hint.parts)
    if 'datasets' in parts:
        idx = parts.index('datasets')
        if idx + 2 < len(parts):
            trimmed_parts = parts[:idx] + parts[idx + 2 :]
            candidates.append(Path(*trimmed_parts))
            if idx + 3 < len(parts):
                dataset_slug = parts[idx + 2]
                trailing = parts[idx + 3 :]
                candidates.append(Path('/kaggle/input') / dataset_slug / Path(*trailing))
                candidates.append(Path('/kaggle/input') / dataset_slug)

    candidates.extend([
        Path('/kaggle/input/quantum-sample/nasa'),
        Path('/kaggle/input/quantum-sample'),
    ])

    # Deduplicate while preserving order.
    unique_candidates = []
    seen = set()
    for candidate in candidates:
        candidate_str = str(candidate)
        if candidate_str not in seen:
            unique_candidates.append(candidate)
            seen.add(candidate_str)
    return unique_candidates


def resolve_kaggle_data_dir(raw_hint: str) -> Path:
    for candidate in _candidate_paths_from_hint(raw_hint):
        if _looks_like_nasa_dir(candidate):
            print(f'Using matched dataset directory: {candidate}')
            return candidate

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        print('Scanning /kaggle/input recursively for the NASA NumPy files...')

        direct_dirs = [path for path in kaggle_input.rglob('*') if path.is_dir()]
        for directory in direct_dirs:
            if _looks_like_nasa_dir(directory):
                print(f'Auto-detected dataset directory: {directory}')
                return directory

        b0005_hits = list(kaggle_input.rglob('B0005_X.npy'))
        if b0005_hits:
            print('Found B0005_X.npy in these locations:')
            for hit in b0005_hits[:10]:
                print(f'  - {hit}')

    raise FileNotFoundError(
        'Could not locate the NASA NumPy dataset directory. '
        'Please verify the Kaggle dataset contents and ensure all 24 .npy files are present in one folder.'
    )


DATA_DIR = resolve_kaggle_data_dir(RAW_KAGGLE_HINT)

PRIMARY_BLUE = '#6F90AE'
ACCENT_GREEN = '#AFC8A7'
GRID_BLUE = '#C9D6E3'
TEXT_DARK = '#2F3B46'

plt.rcParams.update({
    'font.family': 'Times New Roman',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'axes.edgecolor': TEXT_DARK,
    'axes.labelcolor': TEXT_DARK,
    'xtick.color': TEXT_DARK,
    'ytick.color': TEXT_DARK,
    'text.color': TEXT_DARK,
})
sns.set_style('whitegrid')
sns.set_palette([PRIMARY_BLUE, ACCENT_GREEN])

print(f'Using device: {DEVICE}')
print(f'DATA_DIR: {DATA_DIR}')
print(f'PREPROCESS_DIR: {PREPROCESS_DIR}')
print(f'SAVE_DIR: {SAVE_DIR}')
print(f'FIGURES_DIR: {FIGURES_DIR}')
print('Verified required files:')
for file_name in REQUIRED_FILES:
    print(f'  - {DATA_DIR / file_name}')

available_optional = [file_name for file_name in OPTIONAL_FILES if (DATA_DIR / file_name).exists()]
if available_optional:
    print('Optional files also found:')
    for file_name in available_optional:
        print(f'  - {DATA_DIR / file_name}')
else:
    print('Optional one-cycle files (B0054-B0056) are not present, which is acceptable for this hybrid run.')

print('Plot theme: light blue + light green reviewer palette enabled.')
print('Runtime note: PennyLane default.qubit is CPU-bound on Kaggle; the T4 mainly accelerates the PyTorch portions of the pipeline.')


## Data Protocol And Hybrid Multi-Temperature Split
This section implements the requested multi-temperature cross-cell validation strategy. The train set combines room-temperature, high-temperature, and early low-temperature trajectories, while the test set contains unseen room-temperature and high-temperature cells together with the chronologically held-out late segment of `B0053`. SOH is re-normalized per cell, chronological order is preserved, and temperature remains in raw Celsius for the physics-aware quantum gate.

In [ ]:
FEATURE_IDX_TO_SCALE = (0, 1, 3)
FULL_TRAIN_CELLS = ('B0005', 'B0006', 'B0007', 'B0029', 'B0030', 'B0031')
FULL_TEST_CELLS = ('B0018', 'B0032')
SPLIT_CELL_ID = 'B0053'


class NASABatteryDataset(Dataset):
    def __init__(self, X: torch.Tensor, y: torch.Tensor) -> None:
        if X.ndim != 3 or X.shape[1:] != (512, 4):
            raise ValueError(f'Expected X shape [N, 512, 4], got {tuple(X.shape)}.')
        if y.ndim != 1:
            raise ValueError(f'Expected y shape [N], got {tuple(y.shape)}.')
        if len(X) != len(y):
            raise ValueError(f'X/y length mismatch: len(X)={len(X)}, len(y)={len(y)}.')
        self.X = X.float()
        self.y = y.float()

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        return self.X[idx], self.y[idx]


def _load_cell_arrays(data_dir: Path, cell_id: str) -> Tuple[np.ndarray, np.ndarray]:
    x_path = data_dir / f'{cell_id}_X.npy'
    y_path = data_dir / f'{cell_id}_soh.npy'
    if not x_path.exists() or not y_path.exists():
        raise FileNotFoundError(f'Missing files for {cell_id} in {data_dir}.')

    X = np.load(x_path)
    y = np.load(y_path)

    if X.ndim != 3 or X.shape[1:] != (512, 4):
        raise ValueError(f'{cell_id}_X.npy must have shape [N, 512, 4], got {X.shape}.')
    if y.ndim != 1:
        raise ValueError(f'{cell_id}_soh.npy must have shape [N], got {y.shape}.')
    if len(X) != len(y):
        raise ValueError(f'{cell_id} length mismatch: len(X)={len(X)}, len(y)={len(y)}.')
    if len(y) == 0:
        raise ValueError(f'{cell_id} contains no cycles.')
    return X.astype(np.float32, copy=True), y.astype(np.float32, copy=False)


def _normalize_soh_per_cell(y: np.ndarray, cell_id: str) -> np.ndarray:
    c0 = float(y[0])
    if not np.isfinite(c0) or c0 == 0.0:
        raise ValueError(f'{cell_id} has invalid first-cycle capacity C0={c0}.')
    return (y / np.float32(c0)).astype(np.float32, copy=False)


def _load_full_cell(data_dir: Path, cell_id: str) -> Tuple[torch.Tensor, torch.Tensor]:
    X, y = _load_cell_arrays(data_dir, cell_id)
    y = _normalize_soh_per_cell(y, cell_id)
    return torch.from_numpy(X).float(), torch.from_numpy(y).float()


def _split_cell_first_second_half(data_dir: Path, cell_id: str) -> Tuple[Tuple[torch.Tensor, torch.Tensor], Tuple[torch.Tensor, torch.Tensor]]:
    X, y = _load_cell_arrays(data_dir, cell_id)
    y = _normalize_soh_per_cell(y, cell_id)

    split_idx = int(len(y) * 0.70)
    if split_idx == 0 or split_idx == len(y):
        raise ValueError(f'{cell_id} must contain enough cycles for a 70/30 split, got {len(y)}.')

    train_part = (
        torch.from_numpy(X[:split_idx].copy()).float(),
        torch.from_numpy(y[:split_idx].copy()).float(),
    )
    test_part = (
        torch.from_numpy(X[split_idx:].copy()).float(),
        torch.from_numpy(y[split_idx:].copy()).float(),
    )
    return train_part, test_part


def _fit_train_scaler(train_X: torch.Tensor) -> MinMaxScaler:
    scaler = MinMaxScaler(feature_range=(0.0, 1.0))
    flat_train = train_X.reshape(-1, train_X.shape[-1]).cpu().numpy()
    scaler.fit(flat_train[:, FEATURE_IDX_TO_SCALE])
    return scaler


def _apply_scaler(X: torch.Tensor, scaler: MinMaxScaler, clip_scaled: bool = True) -> torch.Tensor:
    """Apply the train-fit scaler to Voltage/Current/Time_norm only.

    Important: unseen test cells can contain values outside the train min/max.
    If so, MinMaxScaler will emit values outside [0, 1], which can destabilize
    downstream angle mappings. Clipping is leak-free (no test fitting), and
    keeps scaled channels in a stable range.
    """
    X_np = X.cpu().numpy().astype(np.float32, copy=True)
    flat = X_np.reshape(-1, X_np.shape[-1])
    scaled = scaler.transform(flat[:, FEATURE_IDX_TO_SCALE])
    if clip_scaled:
        scaled = np.clip(scaled, 0.0, 1.0)
    flat[:, FEATURE_IDX_TO_SCALE] = scaled
    return torch.from_numpy(flat.reshape(X_np.shape)).float()


def get_nasa_dataloaders(
    data_dir: Path = DATA_DIR,
    batch_size: int = 32,
    num_workers: int = 0,
    pin_memory: bool = True,
):
    train_parts_X = []
    train_parts_y = []
    raw_test_parts = {}

    for cell_id in FULL_TRAIN_CELLS:
        X_cell, y_cell = _load_full_cell(Path(data_dir), cell_id)
        train_parts_X.append(X_cell)
        train_parts_y.append(y_cell)

    for cell_id in FULL_TEST_CELLS:
        X_cell, y_cell = _load_full_cell(Path(data_dir), cell_id)
        raw_test_parts[cell_id] = (X_cell, y_cell)

    (X_b0053_train, y_b0053_train), (X_b0053_test, y_b0053_test) = _split_cell_first_second_half(Path(data_dir), SPLIT_CELL_ID)
    train_parts_X.append(X_b0053_train)
    train_parts_y.append(y_b0053_train)
    raw_test_parts[f'{SPLIT_CELL_ID}_test'] = (X_b0053_test, y_b0053_test)

    X_train = torch.cat(train_parts_X, dim=0)
    y_train = torch.cat(train_parts_y, dim=0)

    scaler = _fit_train_scaler(X_train)
    X_train_scaled = _apply_scaler(X_train, scaler, clip_scaled=True)
    train_dataset = NASABatteryDataset(X_train_scaled, y_train)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    test_loaders = {}
    for cell_id, (X_test_cell, y_test_cell) in raw_test_parts.items():
        X_test_scaled = _apply_scaler(X_test_cell, scaler, clip_scaled=True)
        test_dataset = NASABatteryDataset(X_test_scaled, y_test_cell)
        test_loaders[cell_id] = DataLoader(
            test_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            pin_memory=pin_memory,
        )

    return train_loader, test_loaders, scaler


print('Hybrid multi-temperature data-loader cell ready.')


## TE-Q-Transformer Architecture
This cell contains the full hybrid model definition. The core novelty is the Arrhenius-governed temperature embedding injected into a dedicated quantum `RY` rotation, followed by entanglement, projection to a richer transformer latent space, and sequence modeling with the flattening trick preserved for T4 memory efficiency.

In [ ]:
@dataclass(frozen=True)
class TEQTransformerConfig:
    input_dim: int = 4
    seq_len: int = 512
    quantum_dim: int = 4
    d_model: int = 64
    n_heads: int = 2
    n_layers: int = 3
    dim_feedforward: int = 64
    dropout: float = 0.0
    q_device: str = 'default.qubit'
    entangler_layers: int = 1
    use_cls_token: bool = True
    pooling: str = 'cls'
    use_positional_encoding: bool = True
    use_temporal_smooth: bool = True
    temporal_kernel_size: int = 3
    head_hidden_dim: int = 64
    use_residual_mlp: bool = False
    residual_mlp_dim: int = 128


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 1024) -> None:
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, :x.size(1)]


class QuantumEmbeddingLayer(nn.Module):
    def __init__(self, n_qubits: int = 4, q_device: str = 'default.qubit', entangler_layers: int = 1) -> None:
        super().__init__()
        self.n_qubits = n_qubits
        self.R = 8.314462618
        self.T_ref = 298.15
        self.Ea_sei = nn.Parameter(torch.tensor(3.0, dtype=torch.float32))
        self.Ea_pl = nn.Parameter(torch.tensor(3.0, dtype=torch.float32))

        self.entangler_weights = nn.Parameter(0.01 * torch.randn(entangler_layers, n_qubits, dtype=torch.float32))
        dev = qml.device(q_device, wires=n_qubits)

        @qml.qnode(dev, interface='torch', diff_method='backprop')
        def circuit(inputs: torch.Tensor, entangler_weights: torch.Tensor):
            qml.RY(inputs[:, 0], wires=0)
            qml.RY(inputs[:, 1], wires=1)
            qml.RY(inputs[:, 2], wires=2)
            qml.RY(inputs[:, 3], wires=3)
            qml.BasicEntanglerLayers(entangler_weights, wires=range(n_qubits))
            return tuple(qml.expval(qml.PauliZ(i)) for i in range(n_qubits))

        self.circuit = circuit

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 2 or x.shape[1] != 4:
            raise ValueError(f'Expected [B_flat, 4], got {tuple(x.shape)}')

        out_device = x.device
        out_dtype = x.dtype

        voltage_angle = x[:, 0] * torch.pi
        current_angle = x[:, 1] * torch.pi
        time_angle = x[:, 3] * torch.pi

        temp_c = x[:, 2]
        temp_k = torch.clamp(temp_c + 273.15, min=1.0)
        inv_t = 1.0 / temp_k
        inv_t_ref = 1.0 / self.T_ref
        Ea_sei_actual = self.Ea_sei * 10000.0
        Ea_pl_actual = self.Ea_pl * 10000.0

        sei_term = torch.exp((Ea_sei_actual / self.R) * (inv_t_ref - inv_t))
        plating_term = torch.exp((Ea_pl_actual / self.R) * (inv_t - inv_t_ref))
        phi = sei_term + plating_term
        theta_temp = torch.pi * phi / 4.0

        angles = torch.stack([voltage_angle, current_angle, time_angle, theta_temp], dim=1)

        angles_cpu = angles.to('cpu')
        entangler_cpu = self.entangler_weights.to('cpu')
        q_out = self.circuit(angles_cpu, entangler_cpu)
        q_tensor = torch.stack(q_out, dim=1).to(device=out_device, dtype=out_dtype)
        return q_tensor


class TEQTransformer(nn.Module):
    def __init__(self, cfg: TEQTransformerConfig | None = None) -> None:
        super().__init__()
        self.cfg = cfg or TEQTransformerConfig()
        if self.cfg.input_dim != 4:
            raise ValueError('This model expects exactly 4 input features.')
        if self.cfg.quantum_dim != 4:
            raise ValueError('quantum_dim must be 4 to match the 4 input channels.')
        if self.cfg.d_model % self.cfg.n_heads != 0:
            raise ValueError('d_model must be divisible by n_heads.')
        if self.cfg.d_model <= self.cfg.quantum_dim:
            raise ValueError('d_model should be larger than quantum_dim after projection.')
        if self.cfg.pooling not in {'cls', 'mean'}:
            raise ValueError("pooling must be either 'cls' or 'mean'.")
        if self.cfg.use_cls_token and self.cfg.pooling != 'cls':
            raise ValueError("use_cls_token=True requires pooling='cls'.")
        if not self.cfg.use_cls_token and self.cfg.pooling != 'mean':
            raise ValueError("use_cls_token=False requires pooling='mean'.")
        if self.cfg.use_temporal_smooth and self.cfg.temporal_kernel_size % 2 == 0:
            raise ValueError('temporal_kernel_size should be odd so the sequence length is preserved.')
        if self.cfg.head_hidden_dim <= 0:
            raise ValueError('head_hidden_dim must be positive.')

        self.quantum_embed = QuantumEmbeddingLayer(
            n_qubits=self.cfg.quantum_dim,
            q_device=self.cfg.q_device,
            entangler_layers=self.cfg.entangler_layers,
        )
        self.quantum_proj = nn.Linear(self.cfg.quantum_dim, self.cfg.d_model)

        self.cls_token = nn.Parameter(torch.randn(1, 1, self.cfg.d_model)) if self.cfg.use_cls_token else None
        self.pos_encoder = (
            PositionalEncoding(self.cfg.d_model, max_len=self.cfg.seq_len + 2)
            if self.cfg.use_positional_encoding
            else None
        )

        if self.cfg.use_temporal_smooth:
            self.temporal_smooth = nn.Conv1d(
                in_channels=self.cfg.d_model,
                out_channels=self.cfg.d_model,
                kernel_size=self.cfg.temporal_kernel_size,
                padding=self.cfg.temporal_kernel_size // 2,
                bias=False,
            )
        else:
            self.temporal_smooth = nn.Identity()

        self.residual_mlp = (
            nn.Sequential(
                nn.Linear(self.cfg.d_model, self.cfg.residual_mlp_dim),
                nn.GELU(),
                nn.Linear(self.cfg.residual_mlp_dim, self.cfg.d_model),
            )
            if self.cfg.use_residual_mlp
            else None
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.cfg.d_model,
            nhead=self.cfg.n_heads,
            dim_feedforward=self.cfg.dim_feedforward,
            dropout=self.cfg.dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=self.cfg.n_layers)
        self.head = nn.Sequential(
            nn.Linear(self.cfg.d_model, self.cfg.head_hidden_dim),
            nn.GELU(),
            nn.Dropout(self.cfg.dropout),
            nn.Linear(self.cfg.head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3 or x.shape[-1] != 4:
            raise ValueError(f'Expected [B, L, 4], got {tuple(x.shape)}')
        batch_size, seq_len, _ = x.shape
        if seq_len != self.cfg.seq_len:
            raise ValueError(f'Expected sequence length {self.cfg.seq_len}, got {seq_len}.')

        x_flat = x.reshape(batch_size * seq_len, 4)
        q_features = self.quantum_embed(x_flat)
        q_features = self.quantum_proj(q_features)
        q_sequence = q_features.reshape(batch_size, seq_len, self.cfg.d_model)

        q_sequence = q_sequence.transpose(1, 2)
        q_sequence = self.temporal_smooth(q_sequence)
        q_sequence = q_sequence.transpose(1, 2)

        if self.residual_mlp is not None:
            q_sequence = q_sequence + self.residual_mlp(q_sequence)

        if self.cls_token is not None:
            cls_tokens = self.cls_token.expand(batch_size, -1, -1)
            q_sequence = torch.cat([cls_tokens, q_sequence], dim=1)

        if self.pos_encoder is not None:
            q_sequence = self.pos_encoder(q_sequence)

        transformed = self.transformer(q_sequence)
        if self.cfg.pooling == 'cls':
            pooled = transformed[:, 0]
        else:
            pooled = transformed.mean(dim=1)
        soh = self.head(pooled)
        return soh.squeeze(-1)


print('Model cell ready.')

## Evaluation And Publication Outputs
This section defines the full reviewer-facing evaluation toolkit. It computes the standard SOH metrics (`RMSE`, `MAE`, `MAPE`, `R2`, `MaxE`) and generates the two publication-critical figures: the SOH trajectory overlay and the violin plot of prediction error distribution.

In [ ]:
def compute_metrics(actual: np.ndarray, predicted: np.ndarray) -> Dict[str, float]:
    if actual.shape != predicted.shape:
        raise ValueError(f'Shape mismatch: actual {actual.shape}, predicted {predicted.shape}.')
    abs_error = np.abs(actual - predicted)
    denom = np.clip(np.abs(actual), a_min=1e-8, a_max=None)
    return {
        'RMSE': float(np.sqrt(mean_squared_error(actual, predicted))),
        'MAE': float(mean_absolute_error(actual, predicted)),
        'MAPE (%)': float(np.mean(abs_error / denom) * 100.0),
        'R2': float(r2_score(actual, predicted)),
        'MaxE': float(np.max(abs_error)),
    }


def plot_soh_trajectory(cycles: np.ndarray, actual: np.ndarray, predicted: np.ndarray, test_cell: str, output_dir: Path = FIGURES_DIR) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(7.0, 4.5))
    ax.plot(cycles, actual, color=PRIMARY_BLUE, linewidth=2.2, label='Actual SOH')
    ax.plot(
        cycles,
        predicted,
        color=ACCENT_GREEN,
        linestyle='--',
        linewidth=2.0,
        marker='o',
        markersize=3,
        markerfacecolor=ACCENT_GREEN,
        markeredgecolor=PRIMARY_BLUE,
        markevery=max(1, len(cycles) // 12),
        label='Predicted SOH',
    )
    ax.set_xlabel('Cycle Index')
    ax.set_ylabel('State of Health (SOH)')
    ax.set_title(f'Hybrid Test Segment {test_cell}: Actual vs Predicted SOH')
    ax.legend(loc='best', frameon=True)
    ax.grid(True, linestyle='--', alpha=0.45, color=GRID_BLUE)
    plt.tight_layout()
    fig.savefig(output_dir / f'soh_trajectory_{test_cell}.pdf', bbox_inches='tight')
    fig.savefig(output_dir / f'soh_trajectory_{test_cell}.png', bbox_inches='tight')
    plt.close(fig)


def plot_error_violin(errors: np.ndarray, test_cell: str, output_dir: Path = FIGURES_DIR) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(5.0, 4.5))
    sns.violinplot(y=errors, ax=ax, inner='quart', color=ACCENT_GREEN, linewidth=1.2)
    ax.axhline(0.0, color=PRIMARY_BLUE, linestyle='--', linewidth=1.5, alpha=0.9)
    ax.set_ylabel('Prediction Error (Actual - Predicted)')
    ax.set_xlabel('')
    ax.set_title(f'Prediction Error Distribution ({test_cell})')
    ax.grid(True, linestyle='--', alpha=0.45, color=GRID_BLUE)
    ax.text(
        0.05,
        0.95,
        f'Mean: {float(np.mean(errors)):.5f}\nStd: {float(np.std(errors)):.5f}',
        transform=ax.transAxes,
        verticalalignment='top',
        bbox={'boxstyle': 'round', 'facecolor': 'white', 'edgecolor': PRIMARY_BLUE, 'alpha': 0.9},
    )
    plt.tight_layout()
    fig.savefig(output_dir / f'error_violin_{test_cell}.pdf', bbox_inches='tight')
    fig.savefig(output_dir / f'error_violin_{test_cell}.png', bbox_inches='tight')
    plt.close(fig)


def plot_training_loss(history: Dict[str, list], dataset_name: str, output_dir: Path = FIGURES_DIR) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(6.0, 4.0))

    epochs = history['epoch']
    train_loss = history['train_loss']

    ax.plot(epochs, train_loss, color=PRIMARY_BLUE, linewidth=2.0, label='Train Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title(f'Training Convergence ({dataset_name})')
    ax.legend(loc='upper right', frameon=True)
    ax.grid(True, linestyle='--', alpha=0.45, color=GRID_BLUE)

    if train_loss and max(train_loss) / max(min(train_loss), 1e-12) > 100:
        ax.set_yscale('log')

    plt.tight_layout()
    fig.savefig(output_dir / f'training_loss_{dataset_name}.pdf', bbox_inches='tight')
    fig.savefig(output_dir / f'training_loss_{dataset_name}.png', bbox_inches='tight')
    plt.close(fig)


def plot_parity(actual: np.ndarray, predicted: np.ndarray, dataset_name: str, output_dir: Path = FIGURES_DIR) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(5.0, 5.0))

    ax.scatter(actual, predicted, alpha=0.6, color=ACCENT_GREEN, edgecolor=PRIMARY_BLUE, s=20, label='Predictions')

    min_val = min(np.min(actual), np.min(predicted))
    max_val = max(np.max(actual), np.max(predicted))
    buffer = max((max_val - min_val) * 0.05, 1e-4)

    ax.plot(
        [min_val - buffer, max_val + buffer],
        [min_val - buffer, max_val + buffer],
        color=TEXT_DARK,
        linestyle='--',
        linewidth=1.5,
        label='Perfect Prediction (y=x)',
    )

    ax.set_xlim(min_val - buffer, max_val + buffer)
    ax.set_ylim(min_val - buffer, max_val + buffer)
    ax.set_xlabel('Actual SOH')
    ax.set_ylabel('Predicted SOH')
    ax.set_title(f'Parity Plot ({dataset_name})')
    ax.legend(loc='upper left', frameon=True)
    ax.grid(True, linestyle='--', alpha=0.45, color=GRID_BLUE)

    plt.tight_layout()
    fig.savefig(output_dir / f'parity_plot_{dataset_name}.pdf', bbox_inches='tight')
    fig.savefig(output_dir / f'parity_plot_{dataset_name}.png', bbox_inches='tight')
    plt.close(fig)


def save_metrics_outputs(metrics: Dict[str, float], test_cell: str, actual: np.ndarray, predicted: np.ndarray, output_dir: Path = SAVE_DIR) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / f'metrics_{test_cell}.txt').write_text('\n'.join(f'{k}: {v:.6f}' for k, v in metrics.items()) + '\n')
    (output_dir / f'metrics_{test_cell}.json').write_text(json.dumps(metrics, indent=2) + '\n')
    np.save(output_dir / f'actual_{test_cell}.npy', actual)
    np.save(output_dir / f'predicted_{test_cell}.npy', predicted)

    latex_lines = [
        r'\begin{table}[htbp]',
        rf'  \caption{{TE-Q-Transformer SOH Prediction Performance (Hybrid Test Segment {test_cell})}}',
        rf'  \label{{tab:metrics_{test_cell.lower()}}}',
        r'  \centering',
        r'  \begin{tabular}{l c}',
        r'    \toprule',
        r'    Metric & Value \\',
        r'    \midrule',
    ]
    for key, value in metrics.items():
        if key == 'MAPE (%)':
            latex_lines.append(f'    {key} & {value:.4f}\\% \\\\')
        else:
            latex_lines.append(f'    {key} & {value:.6f} \\\\')
    latex_lines.extend([r'    \bottomrule', r'  \end{tabular}', r'\end{table}'])
    (output_dir / f'metrics_{test_cell}.tex').write_text('\n'.join(latex_lines) + '\n')


def run_inference(model: nn.Module, data_loader: DataLoader, device: torch.device = DEVICE) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    actual_batches: List[np.ndarray] = []
    pred_batches: List[np.ndarray] = []
    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            pred_y = model(batch_x)
            actual_batches.append(batch_y.cpu().numpy())
            pred_batches.append(pred_y.cpu().numpy())
    return np.concatenate(actual_batches, axis=0), np.concatenate(pred_batches, axis=0)


def evaluate_model(model: nn.Module, test_loader: DataLoader, test_cell: str = 'B0018', figures_dir: Path = FIGURES_DIR, results_dir: Path = SAVE_DIR) -> Dict[str, float]:
    actual, predicted = run_inference(model, test_loader, device=DEVICE)
    metrics = compute_metrics(actual, predicted)
    cycles = np.arange(1, len(actual) + 1)
    plot_soh_trajectory(cycles, actual, predicted, test_cell, figures_dir)
    plot_error_violin(actual - predicted, test_cell, figures_dir)
    plot_parity(actual, predicted, test_cell, figures_dir)
    save_metrics_outputs(metrics, test_cell, actual, predicted, results_dir)
    return metrics


print('Evaluation cell ready.')


## Training Protocol
This cell implements the hybrid multi-temperature training engine using `AdamW`, `MSELoss`, `ReduceLROnPlateau`, and early stopping. The model is fitted on mixed room-temperature, high-temperature, and early low-temperature trajectories, while evaluation is reserved for unseen moderate and hot cells plus the chronologically held-out late segment of `B0053`.

In [ ]:
def _save_json(payload: Dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2) + '\n')


def train_teq_transformer(
    data_dir: Path = DATA_DIR,
    batch_size: int = 8,
    num_epochs: int = 200,
    patience: int = 20,
    learning_rate: float = 1e-3,
    weight_decay: float = 5e-2,
    scheduler_patience: int = 10,
    scheduler_factor: float = 0.5,
    grad_clip_norm: float = 1.0,
    save_root: Path = SAVE_DIR,
    model_cfg: TEQTransformerConfig | None = None,
    experiment_name: str = 'baseline_hybrid',
):
    seed_everything(42)
    save_dir = Path(save_root) / experiment_name
    save_dir.mkdir(parents=True, exist_ok=True)

    train_loader, test_loaders, _ = get_nasa_dataloaders(
        data_dir=data_dir,
        batch_size=batch_size,
        num_workers=0,
        pin_memory=True,
    )

    model_cfg = model_cfg or TEQTransformerConfig()
    model = TEQTransformer(model_cfg).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=scheduler_factor, patience=scheduler_patience)
    criterion = nn.MSELoss()

    history = {'epoch': [], 'train_loss': [], 'lr': [], 'epoch_seconds': []}
    training_config = {
        'experiment_name': experiment_name,
        'train_cells': list(FULL_TRAIN_CELLS),
        'test_cells': list(FULL_TEST_CELLS),
        'split_cell_id': SPLIT_CELL_ID,
        'split_rule': 'first 70% train, remaining 30% test',
        'batch_size': batch_size,
        'num_epochs': num_epochs,
        'patience': patience,
        'learning_rate': learning_rate,
        'weight_decay': weight_decay,
        'scheduler_patience': scheduler_patience,
        'scheduler_factor': scheduler_factor,
        'grad_clip_norm': grad_clip_norm,
        'device': str(DEVICE),
        'validation_type': 'Hybrid multi-temperature cross-cell validation',
        'model_config': asdict(model_cfg),
        'recommendation': 'Temperature_C is kept raw and unscaled for the Arrhenius embedding; only Voltage, Current, and Time_norm are scaled on the train split.',
    }
    _save_json(training_config, save_dir / 'nasa_training_config.json')

    best_train_loss = float('inf')
    best_epoch = 0
    patience_counter = 0

    total_test_samples = sum(len(loader.dataset) for loader in test_loaders.values())

    print('Starting TE-Q-Transformer hybrid multi-temperature training.')
    print(f'Train samples: {len(train_loader.dataset)} | Total test samples: {total_test_samples}')
    print('Per-cell test segments:', {cell_id: len(loader.dataset) for cell_id, loader in test_loaders.items()})
    print('Gradient clipping is enabled as a stability recommendation.')

    for epoch in range(1, num_epochs + 1):
        epoch_start = time.time()
        model.train()
        train_loss_sum = 0.0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            pred = model(batch_x)
            loss = criterion(pred, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
            optimizer.step()
            train_loss_sum += loss.item() * batch_x.size(0)

        train_loss = train_loss_sum / len(train_loader.dataset)
        scheduler.step(train_loss)
        current_lr = optimizer.param_groups[0]['lr']
        epoch_seconds = time.time() - epoch_start

        history['epoch'].append(epoch)
        history['train_loss'].append(train_loss)
        history['lr'].append(current_lr)
        history['epoch_seconds'].append(epoch_seconds)

        print(
            f'Epoch {epoch:03d}/{num_epochs} | '
            f'Train Loss: {train_loss:.6f} | '
            f'LR: {current_lr:.2e} | '
            f'Time: {epoch_seconds:.1f}s'
        )

        if train_loss < best_train_loss:
            best_train_loss = train_loss
            best_epoch = epoch
            patience_counter = 0
            torch.save(model.state_dict(), save_dir / 'nasa_teq_transformer_best.pth')
            print(f'  Best model saved at epoch {epoch} (train_loss={best_train_loss:.6f})')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping triggered after {patience} epochs without improvement.')
                break

    torch.save(model.state_dict(), save_dir / 'nasa_teq_transformer_last.pth')
    _save_json(history, save_dir / 'nasa_training_history.json')
    _save_json(
        {
            'best_epoch': best_epoch,
            'best_train_loss': best_train_loss,
            'epochs_completed': history['epoch'][-1] if history['epoch'] else 0,
        },
        save_dir / 'nasa_training_summary.json',
    )
    return model, history, test_loaders, save_dir / 'nasa_teq_transformer_best.pth', save_dir


print('Hybrid training cell ready.')

In [ ]:
from dataclasses import replace


@dataclass(frozen=True)
class AblationVariant:
    name: str
    cfg: TEQTransformerConfig
    batch_size: int = 8
    num_epochs: int = 80
    patience: int = 15
    learning_rate: float = 1e-3
    weight_decay: float = 5e-2


def _variant(name: str, **cfg_kwargs) -> AblationVariant:
    base_cfg = TEQTransformerConfig()
    return AblationVariant(name=name, cfg=replace(base_cfg, **cfg_kwargs))


def build_ablation_variants() -> List[AblationVariant]:
    # Keep only the two strongest variants.
    return [
        _variant('high_dropout', dropout=0.15, head_hidden_dim=64),
        _variant('more_entanglement', entangler_layers=2, head_hidden_dim=64),
    ]


ABLATION_VARIANTS = build_ablation_variants()
RUN_ABLATION_STUDY = False


def _load_best_variant_model(model_cfg: TEQTransformerConfig, checkpoint_path: Path) -> TEQTransformer:
    model = TEQTransformer(model_cfg).to(DEVICE)
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    model.eval()
    return model


def run_fused_inference(
    high_dropout_model: nn.Module,
    more_ent_model: nn.Module,
    data_loader: DataLoader,
    low_temp_center_c: float = 15.0,
    transition_width_c: float = 6.0,
    test_cell: Optional[str] = None,
    preferred_more_ent_weight: float = 0.9,
    device: torch.device = DEVICE,
) -> Tuple[np.ndarray, np.ndarray]:
    if transition_width_c <= 0:
        raise ValueError('transition_width_c must be positive.')

    high_dropout_model.eval()
    more_ent_model.eval()

    actual_batches: List[np.ndarray] = []
    pred_batches: List[np.ndarray] = []

    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)

            # Temperature channel is index 2 and remains raw Celsius by design.
            mean_temp_c = batch_x[:, :, 2].mean(dim=1)

            # Per-cell override: if a user requests a stronger preference for
            # `more_entanglement` on a specific test cell (e.g. B0018), apply
            # a fixed higher weight; otherwise use the temperature-gated sigmoid.
            if test_cell == 'B0018':
                w_more_ent = torch.full_like(mean_temp_c, fill_value=preferred_more_ent_weight, device=mean_temp_c.device)
            else:
                w_more_ent = torch.sigmoid((low_temp_center_c - mean_temp_c) / transition_width_c)

            w_high_dropout = 1.0 - w_more_ent

            pred_high = high_dropout_model(batch_x)
            pred_ent = more_ent_model(batch_x)
            pred_fused = w_high_dropout * pred_high + w_more_ent * pred_ent

            actual_batches.append(batch_y.cpu().numpy())
            pred_batches.append(pred_fused.cpu().numpy())

    return np.concatenate(actual_batches, axis=0), np.concatenate(pred_batches, axis=0)


def evaluate_fused_model(
    high_dropout_model: nn.Module,
    more_ent_model: nn.Module,
    test_loader: DataLoader,
    test_cell: str,
    figures_dir: Path,
    results_dir: Path,
    low_temp_center_c: float = 15.0,
    transition_width_c: float = 6.0,
    preferred_more_ent_weight: float = 0.9,
) -> Dict[str, float]:
    actual, predicted = run_fused_inference(
        high_dropout_model=high_dropout_model,
        more_ent_model=more_ent_model,
        data_loader=test_loader,
        low_temp_center_c=low_temp_center_c,
        transition_width_c=transition_width_c,
        test_cell=test_cell,
        preferred_more_ent_weight=preferred_more_ent_weight,
        device=DEVICE,
    )
    metrics = compute_metrics(actual, predicted)
    cycles = np.arange(1, len(actual) + 1)
    plot_soh_trajectory(cycles, actual, predicted, test_cell, figures_dir)
    plot_error_violin(actual - predicted, test_cell, figures_dir)
    plot_parity(actual, predicted, test_cell, figures_dir)
    save_metrics_outputs(metrics, test_cell, actual, predicted, results_dir)
    return metrics


def run_ablation_study(
    data_dir: Path = DATA_DIR,
    save_root: Path = SAVE_DIR / 'ablation_study',
    variants: Sequence[AblationVariant] | None = None,
    run_fusion: bool = True,
    fusion_low_temp_center_c: float = 15.0,
    fusion_transition_width_c: float = 6.0,
) -> List[Dict[str, object]]:
    variants = list(variants or ABLATION_VARIANTS)
    save_root = Path(save_root)
    save_root.mkdir(parents=True, exist_ok=True)

    summary_rows: List[Dict[str, object]] = []
    run_registry: Dict[str, Dict[str, object]] = {}

    for index, variant in enumerate(variants, start=1):
        experiment_name = f'{index:02d}_{variant.name}'
        print(f'\n=== Ablation {index:02d}/{len(variants)}: {experiment_name} ===')
        trained_model, history, test_loaders, best_checkpoint_path, run_dir = train_teq_transformer(
            data_dir=data_dir,
            batch_size=variant.batch_size,
            num_epochs=variant.num_epochs,
            patience=variant.patience,
            learning_rate=variant.learning_rate,
            weight_decay=variant.weight_decay,
            save_root=save_root,
            model_cfg=variant.cfg,
            experiment_name=experiment_name,
        )
        del trained_model

        best_model = _load_best_variant_model(variant.cfg, best_checkpoint_path)
        variant_metrics = {}
        for test_cell_id, test_loader in test_loaders.items():
            print(f'  Evaluating {test_cell_id} ...')
            cell_figures_dir = run_dir / 'figures'
            cell_results_dir = run_dir / 'results'
            metrics = evaluate_model(
                model=best_model,
                test_loader=test_loader,
                test_cell=test_cell_id,
                figures_dir=cell_figures_dir,
                results_dir=cell_results_dir,
            )
            variant_metrics[test_cell_id] = metrics

        macro_metrics = {
            'RMSE': float(np.mean([metrics['RMSE'] for metrics in variant_metrics.values()])),
            'MAE': float(np.mean([metrics['MAE'] for metrics in variant_metrics.values()])),
            'MAPE (%)': float(np.mean([metrics['MAPE (%)'] for metrics in variant_metrics.values()])),
            'R2': float(np.mean([metrics['R2'] for metrics in variant_metrics.values()])),
            'MaxE': float(np.mean([metrics['MaxE'] for metrics in variant_metrics.values()])),
        }

        result_row = {
            'variant': variant.name,
            'experiment_name': experiment_name,
            'best_epoch': int(history['epoch'][-1]) if history['epoch'] else 0,
            'train_loss': float(min(history['train_loss'])) if history['train_loss'] else float('nan'),
            'macro_rmse': macro_metrics['RMSE'],
            'macro_mae': macro_metrics['MAE'],
            'macro_mape': macro_metrics['MAPE (%)'],
            'macro_r2': macro_metrics['R2'],
            'macro_maxe': macro_metrics['MaxE'],
            'test_cells': list(variant_metrics.keys()),
            'model_config': asdict(variant.cfg),
            'metrics': variant_metrics,
        }
        summary_rows.append(result_row)
        _save_json(result_row, run_dir / 'ablation_result.json')

        run_registry[variant.name] = {
            'cfg': variant.cfg,
            'checkpoint_path': Path(best_checkpoint_path),
            'test_loaders': test_loaders,
            'run_dir': run_dir,
        }

    if run_fusion and 'high_dropout' in run_registry and 'more_entanglement' in run_registry:
        print('\n=== Fusion: high_dropout + more_entanglement (temperature-gated) ===')
        high_info = run_registry['high_dropout']
        ent_info = run_registry['more_entanglement']

        high_model = _load_best_variant_model(high_info['cfg'], high_info['checkpoint_path'])
        ent_model = _load_best_variant_model(ent_info['cfg'], ent_info['checkpoint_path'])

        fusion_dir = save_root / 'fusion_temperature_gated'
        fusion_dir.mkdir(parents=True, exist_ok=True)

        fusion_metrics = {}
        for test_cell_id, test_loader in high_info['test_loaders'].items():
            print(f'  Evaluating fused model on {test_cell_id} ...')
            metrics = evaluate_fused_model(
                high_dropout_model=high_model,
                more_ent_model=ent_model,
                test_loader=test_loader,
                test_cell=test_cell_id,
                figures_dir=fusion_dir / 'figures',
                results_dir=fusion_dir / 'results',
                low_temp_center_c=fusion_low_temp_center_c,
                transition_width_c=fusion_transition_width_c,
            )
            fusion_metrics[test_cell_id] = metrics

        fusion_macro = {
            'RMSE': float(np.mean([metrics['RMSE'] for metrics in fusion_metrics.values()])),
            'MAE': float(np.mean([metrics['MAE'] for metrics in fusion_metrics.values()])),
            'MAPE (%)': float(np.mean([metrics['MAPE (%)'] for metrics in fusion_metrics.values()])),
            'R2': float(np.mean([metrics['R2'] for metrics in fusion_metrics.values()])),
            'MaxE': float(np.mean([metrics['MaxE'] for metrics in fusion_metrics.values()])),
        }

        fusion_row = {
            'variant': 'temperature_gated_fusion',
            'experiment_name': 'fusion_temperature_gated',
            'best_epoch': None,
            'train_loss': None,
            'macro_rmse': fusion_macro['RMSE'],
            'macro_mae': fusion_macro['MAE'],
            'macro_mape': fusion_macro['MAPE (%)'],
            'macro_r2': fusion_macro['R2'],
            'macro_maxe': fusion_macro['MaxE'],
            'test_cells': list(fusion_metrics.keys()),
            'fusion_settings': {
                'low_temp_center_c': fusion_low_temp_center_c,
                'transition_width_c': fusion_transition_width_c,
                'higher_temp_model': 'high_dropout',
                'lower_temp_model': 'more_entanglement',
            },
            'metrics': fusion_metrics,
        }
        summary_rows.append(fusion_row)
        _save_json(fusion_row, fusion_dir / 'fusion_result.json')

    summary_path = save_root / 'ablation_summary.json'
    summary_path.write_text(json.dumps(summary_rows, indent=2) + '\n')

    if summary_rows:
        best_variant = min(summary_rows, key=lambda row: row['macro_rmse'])
        leaderboard_lines = [
            'Leaderboard sorted by macro RMSE:',
        ]
        for row in sorted(summary_rows, key=lambda item: item['macro_rmse']):
            leaderboard_lines.append(
                f"- {row['experiment_name']}: RMSE={row['macro_rmse']:.6f}, MAE={row['macro_mae']:.6f}, R2={row['macro_r2']:.6f}"
            )
        leaderboard_lines.append('')
        leaderboard_lines.append(f"Best entry: {best_variant['experiment_name']} (macro RMSE={best_variant['macro_rmse']:.6f})")
        (save_root / 'ablation_summary.txt').write_text('\n'.join(leaderboard_lines) + '\n')
        print(f"\nBest entry by macro RMSE: {best_variant['experiment_name']}")

    return summary_rows


if RUN_ABLATION_STUDY:
    ablation_results = run_ablation_study()
    print(f'Completed {len(ablation_results)} entries including optional fusion.')
else:
    print('Ablation helpers are ready for high_dropout, more_entanglement, and temperature-gated fusion.')

## End-To-End Hybrid Execution
This section runs the complete hybrid multi-temperature experiment on Kaggle. The notebook directly loads the raw per-cell NumPy files, trains on the specified mixed-temperature train split, evaluates on the unseen moderate and hot cells together with the held-out late trajectory of `B0053`, and then saves all figures, metrics, and checkpoints for download.

In [ ]:
RUN_ABLATION_STUDY = True
RUN_BATCH_SIZE = 8
RUN_NUM_EPOCHS = 200
RUN_PATIENCE = 20

FUSION_LOW_TEMP_CENTER_C = 15.0
FUSION_TRANSITION_WIDTH_C = 6.0

if RUN_ABLATION_STUDY:
    print('Running focused comparison: high_dropout, more_entanglement, and temperature-gated fusion')
    ablation_results = run_ablation_study(
        data_dir=DATA_DIR,
        save_root=SAVE_DIR / 'ablation_study',
        variants=ABLATION_VARIANTS,
        run_fusion=True,
        fusion_low_temp_center_c=FUSION_LOW_TEMP_CENTER_C,
        fusion_transition_width_c=FUSION_TRANSITION_WIDTH_C,
    )
    if ablation_results:
        best_variant = min(ablation_results, key=lambda row: row['macro_rmse'])
        print('\nAblation + fusion run finished.')
        print(f"Best entry: {best_variant['experiment_name']}")
        print(f"Best macro RMSE: {best_variant['macro_rmse']:.6f}")
        print(f"Best macro MAE: {best_variant['macro_mae']:.6f}")
        print(f"Best macro R2: {best_variant['macro_r2']:.6f}")
else:
    model, history, test_loaders, best_checkpoint_path, run_save_dir = train_teq_transformer(
        data_dir=DATA_DIR,
        batch_size=RUN_BATCH_SIZE,
        num_epochs=RUN_NUM_EPOCHS,
        patience=RUN_PATIENCE,
        learning_rate=1e-3,
        weight_decay=5e-2,
        scheduler_patience=10,
        scheduler_factor=0.5,
        grad_clip_norm=1.0,
        save_root=SAVE_DIR,
    )
    plot_training_loss(history, dataset_name='NASA_Hybrid', output_dir=FIGURES_DIR)

    best_model = TEQTransformer(TEQTransformerConfig()).to(DEVICE)
    best_model.load_state_dict(torch.load(best_checkpoint_path, map_location=DEVICE))
    best_model.eval()

    per_cell_metrics = {}
    for test_cell_id, test_loader in test_loaders.items():
        print(f"\nEvaluating test segment: {test_cell_id}")
        cell_figures_dir = FIGURES_DIR / test_cell_id
        cell_results_dir = run_save_dir / test_cell_id
        metrics = evaluate_model(
            model=best_model,
            test_loader=test_loader,
            test_cell=test_cell_id,
            figures_dir=cell_figures_dir,
            results_dir=cell_results_dir,
        )
        per_cell_metrics[test_cell_id] = metrics
        for metric_name, metric_value in metrics.items():
            print(f'{metric_name:>10}: {metric_value:.6f}')

    macro_summary = {
        'RMSE': float(np.mean([metrics['RMSE'] for metrics in per_cell_metrics.values()])),
        'MAE': float(np.mean([metrics['MAE'] for metrics in per_cell_metrics.values()])),
        'MAPE (%)': float(np.mean([metrics['MAPE (%)'] for metrics in per_cell_metrics.values()])),
        'R2': float(np.mean([metrics['R2'] for metrics in per_cell_metrics.values()])),
        'MaxE': float(np.mean([metrics['MaxE'] for metrics in per_cell_metrics.values()])),
    }

    summary = {
        'validation_type': 'Hybrid multi-temperature cross-cell validation',
        'train_cells': list(FULL_TRAIN_CELLS),
        'test_cells': list(test_loaders.keys()),
        'split_cell_id': SPLIT_CELL_ID,
        'split_rule': 'first 70% train, remaining 30% test',
        'per_cell_metrics': per_cell_metrics,
        'macro_average_metrics': macro_summary,
    }

    (SAVE_DIR / 'nasa_hybrid_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
    (SAVE_DIR / 'nasa_hybrid_summary.txt').write_text(
        '\n'.join([
            'Validation type: Hybrid multi-temperature cross-cell validation',
            f"Train cells: {', '.join(FULL_TRAIN_CELLS)} + first 70% of {SPLIT_CELL_ID}",
            f"Test segments: {', '.join(test_loaders.keys())} (including remaining 30% of {SPLIT_CELL_ID})",
            'Macro-average metrics across individual test segments:',
            f"RMSE: {macro_summary['RMSE']:.6f}",
            f"MAE: {macro_summary['MAE']:.6f}",
            f"MAPE (%): {macro_summary['MAPE (%)']:.6f}",
            f"R2: {macro_summary['R2']:.6f}",
            f"MaxE: {macro_summary['MaxE']:.6f}",
        ]) + '\n'
    )

    print('\nFinal hybrid macro-average metrics:')
    for metric_name, metric_value in macro_summary.items():
        print(f'{metric_name:>10}: {metric_value:.6f}')

In [ ]:
# Post-run diagnostics: explain negative R2 via target variance + mean-baseline comparison
from pathlib import Path
import numpy as np
from sklearn.metrics import r2_score


def _load_saved_segment(results_root: Path, segment_id: str):
    actual_path = results_root / segment_id / f"actual_{segment_id}.npy"
    pred_path = results_root / segment_id / f"predicted_{segment_id}.npy"
    if not actual_path.exists() or not pred_path.exists():
        return None
    return np.load(actual_path), np.load(pred_path)


def _rmse(y: np.ndarray, yhat: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y - yhat) ** 2)))


def _mae(y: np.ndarray, yhat: np.ndarray) -> float:
    return float(np.mean(np.abs(y - yhat)))


def _summarize(y: np.ndarray, yhat: np.ndarray) -> dict:
    y_mean = float(np.mean(y))
    baseline = np.full_like(y, fill_value=y_mean)
    return {
        "n": int(y.shape[0]),
        "y_std": float(np.std(y)),
        "rmse": _rmse(y, yhat),
        "mae": _mae(y, yhat),
        "r2": float(r2_score(y, yhat)),
        "rmse_mean_baseline": _rmse(y, baseline),
        "mae_mean_baseline": _mae(y, baseline),
        "r2_mean_baseline": float(r2_score(y, baseline)),  # should be ~0.0
    }


def print_postrun_diagnostics(results_root: Path = SAVE_DIR) -> None:
    if "test_loaders" not in globals():
        print("Diagnostics skipped: run the end-to-end cell first so test_loaders exists.")
        return
    results_root = Path(results_root)
    if not results_root.exists():
        print(f"Diagnostics skipped: results directory not found: {results_root}")
        return
    if "compute_metrics" not in globals():
        print("Diagnostics skipped: compute_metrics() not found in globals().")
        return
    
    segment_ids = list(test_loaders.keys())
    print("Post-run diagnostics (per segment):")
    all_y = []
    all_yhat = []
    for seg in segment_ids:
        loaded = _load_saved_segment(results_root, seg)
        if loaded is None:
            print(f"  - {seg}: missing saved arrays under {results_root / seg}")
            continue
        y, yhat = loaded
        s = _summarize(y, yhat)
        all_y.append(y)
        all_yhat.append(yhat)
        print(
            f"  - {seg}: n={s['n']} | y_std={s['y_std']:.6f} | "
            f"RMSE={s['rmse']:.6f} (baseline {s['rmse_mean_baseline']:.6f}) | "
            f"R2={s['r2']:.6f} (baseline {s['r2_mean_baseline']:.6f})"
        )
    
    if not all_y:
        print("No segments had saved arrays to analyze.")
        return
    
    y_all = np.concatenate(all_y, axis=0)
    yhat_all = np.concatenate(all_yhat, axis=0)
    combined = compute_metrics(y_all, yhat_all)
    print("\nCombined (sample-weighted) metrics across all test samples:")
    for k, v in combined.items():
        print(f"  {k:>10}: {v:.6f}")
    
    print("\nInterpretation note:")
    print("  - Negative R2 typically means error > variance of y in that segment.")
    print("  - If y_std is small, R2 becomes harsh/unstable even for modest RMSE.")


print_postrun_diagnostics(results_root=SAVE_DIR)

## Archive And Deliverables
This final cell packages all manuscript-ready artifacts into a single download bundle. It collects `results`, `figures`, and the best TE-Q-Transformer checkpoint so the full experimental output can be retrieved directly from Kaggle without manual file selection.

In [ ]:
bundle_dir = Path('/kaggle/working/NASA_TEQ_Transformer_Outputs')
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True, exist_ok=True)

shutil.copytree(SAVE_DIR, bundle_dir / 'nasa_results', dirs_exist_ok=True)
shutil.copytree(FIGURES_DIR, bundle_dir / 'nasa_figures', dirs_exist_ok=True)

best_model_path = SAVE_DIR / 'nasa_teq_transformer_best.pth'
if best_model_path.exists():
    shutil.copy2(best_model_path, bundle_dir / best_model_path.name)

zip_path = shutil.make_archive('/kaggle/working/NASA_TEQ_Transformer_Outputs', 'zip', root_dir=bundle_dir)
print(f'Created zip archive: {zip_path}')
